# 1. Import and Hardware Setup

In [96]:
import torch
import torch.nn as nn
import torch.optim as optim

from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split, Subset

!pip install tqdm ipywidgets -q
from tqdm.auto import tqdm 


# 2. Hyperparameter

In [97]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

DATA_PATH = './Data'

cuda


In [98]:
IMG_SIZE = 224
IN_CHANNELS = 3
BATCH_SIZE = 128
NUM_CLASSES = 101

EPOCHS = 150
LR = 0.2
COMPRESSION = 0.5
DROP_RATE = 0.2
BN_SIZE = 4

In [99]:
stats = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(IMG_SIZE),
    transforms.AutoAugment(transforms.AutoAugmentPolicy.IMAGENET),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(*stats),
    transforms.RandomErasing(p=0.5, scale=(0.02, 0.33), ratio=(0.3, 3.3), value='random')
])

test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(*stats)
])

In [100]:
# Download train data first without transform
dummy_data = datasets.Food101(root=DATA_PATH, split='train', download=True)

# Split the train data into train and validation data
train_size = int(0.8 * len(dummy_data))
val_size = len(dummy_data) - train_size
train_subset_tmp, val_subset_tmp = random_split(dummy_data, [train_size, val_size])

# Extract indices from them
train_idx = train_subset_tmp.indices
val_idx = val_subset_tmp.indices

# Create Subsets using the indices
train_dataset = datasets.Food101(root=DATA_PATH, split='train',
                                download=False, transform=train_transform)
val_dataset = datasets.Food101(root=DATA_PATH, split='train',
                                download=False, transform=test_transform)

# Create Subsets using the indices
train_subset = Subset(train_dataset, train_idx)
val_subset = Subset(val_dataset, val_idx)

# Download Test Dataset
test_dataset = datasets.Food101(root=DATA_PATH, split='test',
                                download=True, transform=test_transform)

In [101]:
train_loader = DataLoader(train_subset, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=8, pin_memory=True, persistent_workers=True, prefetch_factor=2)

val_loader = DataLoader(val_subset, batch_size=BATCH_SIZE, shuffle=False,
                        num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False,
                         num_workers=4, pin_memory=True, persistent_workers=True, prefetch_factor=2)

# 4. Network architecture (DenseNet-BC)
This implements the DenseNet-BC variant, featuring Bottleneck layers (1x1 convs in DenseLayers) and Compression (0.5 reduction in TransitionLayers).

![DenseNer](DenseNet.png)

In [102]:
class DenseLayer(nn.Module):
    def __init__(self, in_channels, growth_rate, bn_size=4, drop_rate=0.0):
        super().__init__()
        inter_channels = bn_size * growth_rate
        
        self.bn1 = nn.BatchNorm2d(in_channels)
        self.relu1 = nn.ReLU(inplace=True)
        self.conv1 = nn.Conv2d(in_channels, inter_channels, kernel_size=1, bias=False)
        
        self.bn2 = nn.BatchNorm2d(inter_channels)
        self.relu2 = nn.ReLU(inplace=True)
        self.conv2 = nn.Conv2d(inter_channels, growth_rate, kernel_size=3, padding=1, bias=False)
        
        self.dropout = nn.Dropout2d(p=drop_rate) if drop_rate > 0 else None

    def forward(self, x):
        out = self.conv1(self.relu1(self.bn1(x)))
        out = self.conv2(self.relu2(self.bn2(out)))
        if self.dropout is not None:
            out = self.dropout(out)
        return torch.cat([x, out], 1)

class DenseBlock(nn.Module):
    def __init__(self, num_layers, in_channels, growth_rate, bn_size=4, drop_rate=0.0):
        super().__init__()
        layers = []
        channels = in_channels
        for _ in range(num_layers):
            layers.append(DenseLayer(channels, growth_rate, bn_size, drop_rate))
            channels += growth_rate
        self.block = nn.Sequential(*layers)
        self.out_channels = channels

    def forward(self, x):
        return self.block(x)

class TransitionLayer(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.bn = nn.BatchNorm2d(in_channels)
        self.relu = nn.ReLU(inplace=True)
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False)
        self.pool = nn.AvgPool2d(kernel_size=2, stride=2)

    def forward(self, x):
        x = self.conv(self.relu(self.bn(x)))
        return self.pool(x)

class DenseNet(nn.Module):
    """
    DenseNet-BC implementation:
    - B: Bottleneck layers with 1x1 convolutions before 3x3 convolutions.
    - C: Compression at Transition layers to reduce the number of feature maps.
    """
    def __init__(self, growth_rate=32, block_layers=(6, 12, 24, 16), num_classes=NUM_CLASSES,
                 compression=0.5, bn_size=4, drop_rate=0.2, num_init_features=64):
        super().__init__()
        self.conv1 = nn.Sequential(
            nn.Conv2d(IN_CHANNELS, num_init_features, kernel_size=7, stride=2, padding=3, bias=False),
            nn.BatchNorm2d(num_init_features),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=3, stride=2, padding=1),
        )

        channels = num_init_features
        self.blocks = nn.ModuleList()
        for i, num_layers in enumerate(block_layers):
            block = DenseBlock(num_layers, channels, growth_rate, bn_size, drop_rate)
            self.blocks.append(block)
            channels = block.out_channels
            if i != len(block_layers) - 1:
                out_channels = int(channels * compression)
                trans = TransitionLayer(channels, out_channels)
                self.blocks.append(trans)
                channels = out_channels

        self.bn = nn.BatchNorm2d(channels)
        self.relu = nn.ReLU(inplace=True)
        self.avg_pool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(channels, num_classes)

        self._initialize_weights()

    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight)
            elif isinstance(m, nn.BatchNorm2d):
                nn.init.constant_(m.weight, 1)
                nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.constant_(m.bias, 0)

    def forward(self, x):
        x = self.conv1(x)
        for block in self.blocks:
            x = block(x)
        x = self.relu(self.bn(x))
        x = self.avg_pool(x)
        x = torch.flatten(x, 1)
        return self.fc(x)

def DenseNet121(num_classes=NUM_CLASSES):
    return DenseNet(growth_rate=32, block_layers=(6, 12, 24, 16), num_classes=num_classes,
                   compression=COMPRESSION, bn_size=BN_SIZE, drop_rate=DROP_RATE)

In [103]:
model = DenseNet121(num_classes=NUM_CLASSES).to(device)
print(f"Total parameters: {(sum(p.numel() for p in model.parameters())/1e6):.2f}M")

# Use multiple GPUs if available
if torch.cuda.device_count() > 1:
    print(f"Using {torch.cuda.device_count()} GPUs")
    model = nn.DataParallel(model)

Total parameters: 7.06M


# 5. Training Preparation

In [104]:
class EarlyStopping(nn.Module):
    def __init__(self, patience=10, delta=0, verbose=False, save_path='best_checkpoint.pth'):
        self.patience = patience
        self.delta = delta
        self.verbose = verbose
        self.save_path = save_path
        self.best_loss = None
        self.early_stop = False
        self.counter = 0

    def __call__(self, val_loss, model):
        if self.best_loss is None:
            self.best_loss = val_loss
            self.save_checkpoint(model)
        elif val_loss > self.best_loss - self.delta:
            self.counter += 1
            print(f"Early Stopping counter: {self.counter} out of {self.patience}")
            if self.counter > self.patience:
                self.early_stop = True
        else:
            self.counter = 0
            self.best_loss = val_loss
            self.save_checkpoint(model)

    def save_checkpoint(self, model):
        if self.verbose:
            print("Saving best checkpoint ... ")
        state_dict = model.module.state_dict() if hasattr(model, 'module') else model.state_dict()
        torch.save(state_dict, self.save_path)

In [105]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=LR, momentum=0.9, weight_decay=1e-4, nesterov=True)
scheduler = optim.lr_scheduler.MultiStepLR(
    optimizer, milestones=[int(EPOCHS * 0.5), int(EPOCHS * 0.75)], gamma=0.1
)

scaler = torch.amp.GradScaler(device)

# 6. Train

In [106]:
def train(model, loader, criterion, optimizer):
    model.train()
    train_loss, train_acc = 0, 0
    loop = tqdm(loader, desc='Training', leave=False)
    
    for x, y in loop:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(device_type=device.type):
            out = model(x)
            loss = criterion(out, y)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        train_loss += loss.detach() * x.size(0)
        train_acc += (out.argmax(1) == y).sum()
    return train_loss.item() / len(loader.dataset), train_acc.item() / len(loader.dataset)

def validate(model, loader, criterion):
    model.eval()
    val_loss = 0
    loop = tqdm(loader, desc='Validation', leave=False)

    for x, y in loop:
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            out = model(x)
            loss = criterion(out, y)
        val_loss += loss.detach() * x.size(0)
    return val_loss.item() / len(loader.dataset)

def test(model, loader):
    model.eval()
    test_acc = 0
    loop = tqdm(loader, desc='Testing', leave=False)

    for x, y in loop:
        x, y = x.to(device), y.to(device)
        with torch.no_grad():
            out = model(x)
        test_acc += (out.argmax(1) == y).sum()
    return test_acc.item() / len(loader.dataset)

In [107]:
train_losses, val_losses = [], []
train_accuracies, test_accuracies = [], []
early_stopping = EarlyStopping(patience=10)

for epoch in range(EPOCHS):
    train_loss, train_acc = train(model, train_loader, criterion, optimizer)
    val_loss = validate(model, val_loader, criterion)
    test_acc = test(model, test_loader)
    
    scheduler.step()
    
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)
    
    print(f"Epoch {epoch+1}/{EPOCHS}: train_loss: {train_loss:.2f}, val_loss: {val_loss:.2f}, " +
          f"train_acc: {train_acc:.2f}, test_acc: {test_acc:.2f}")
    
    early_stopping(val_loss, model)
    if early_stopping.early_stop:
        print("Early Stopping")
        break

Training:   0%|          | 0/474 [00:00<?, ?it/s]

Validation:   0%|          | 0/119 [00:00<?, ?it/s]

Testing:   0%|          | 0/198 [00:00<?, ?it/s]

Epoch 1/150: train_loss: 4.50, val_loss: 4.31, train_acc: 0.03, test_acc: 0.06


Training:   0%|          | 0/474 [00:00<?, ?it/s]

...

Validation:   0%|          | 0/119 [00:00<?, ?it/s]

Testing:   0%|          | 0/198 [00:00<?, ?it/s]

Epoch 40/150: train_loss: 1.97, val_loss: 1.90, train_acc: 0.50, test_acc: 0.57
Early Stopping counter: 4 out of 10


Training:   0%|          | 0/474 [00:00<?, ?it/s]

KeyboardInterrupt: 

# 7. GradCAM

In [ ]:
import torch.nn.functional as F
import matplotlib.pyplot as plt

class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer
        self.activations = None
        self.gradients = None
        self._register_hooks()

    def _register_hooks(self):
        def forward_hook(_, __, output):
            self.activations = output

        def backward_hook(_, grad_input, grad_output):
            self.gradients = grad_output[0]

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_full_backward_hook(backward_hook)

    def generate_cam(self, x, class_idx=None):
        self.model.zero_grad(set_to_none=True)
        scores = self.model(x)
        if class_idx is None:
            class_idx = scores.argmax(dim=1)
        score = scores.gather(1, class_idx.view(-1, 1)).squeeze()
        score.backward(retain_graph=True)

        weights = self.gradients.mean(dim=(2, 3), keepdim=True)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)
        cam = F.relu(cam)
        cam = F.interpolate(cam, size=x.shape[-2:], mode='bilinear', align_corners=False)
        cam = cam - cam.min(dim=-1, keepdim=True)[0].min(dim=-2, keepdim=True)[0]
        cam = cam / (cam.max(dim=-1, keepdim=True)[0].max(dim=-2, keepdim=True)[0] + 1e-6)
        return cam

def denorm(img, mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)):
    mean = torch.tensor(mean, device=img.device).view(1, -1, 1, 1)
    std = torch.tensor(std, device=img.device).view(1, -1, 1, 1)
    return img * std + mean

base_model = model.module if hasattr(model, 'module') else model
target_layer = base_model.blocks[-1]
gradcam = GradCAM(base_model, target_layer)

images, labels = next(iter(test_loader))
images = images.to(device)[:1]
cams = gradcam.generate_cam(images)

img = denorm(images).detach().cpu()[0].permute(1, 2, 0).clamp(0, 1)
heatmap = cams.detach().cpu()[0, 0]

plt.figure(figsize=(6, 6))
plt.imshow(img)
plt.imshow(heatmap, cmap='jet', alpha=0.4)
plt.axis('off')
plt.title('GradCAM')